# PMCO | 다음 단계 실습 노트북
### ① v2 가중치 실험 확인 → ② Commons 크롤링·특징 분석 → ③ 사람 평가·가중치 검증

**이 노트북은 기존 `PMCO_Colab_v2.ipynb` 및 `pmco_evaluation.csv`를 덮어쓰지 않는 보조 실습 자료입니다.**

- ①은 기존 6장만으로 즉시 실행 가능. **유료 Gemini/Lyria 호출 없음.**
- ②는 Wikimedia Commons API에서 메타데이터와 작은 썸네일을 가져와 새 CSV 파일로 저장. 크롤링 사이트 정책/속도 제한 준수.
- ③은 사용자가 실제 사진을 보고 점수를 입력한 *뒤* 실행. 음악 적합도 평가에는 실제 음악을 들어야 합니다.
- **오래된 실행 결과와 CSV가 섞이지 않도록** 결과 파일을 별도 이름으로 저장합니다.
- Colab의 `파일 → GitHub에 사본 저장`은 `.ipynb`만 저장합니다. CSV/`.py`는 별도 업로드하세요.

## ①-1. 준비: v2 핵심 모듈과 CSV 올리기

새 Colab에서 이 노트북을 열었다면 `pmco_core_v2.py`와 `pmco_evaluation.csv` 2개를 업로드하세요.
기존 v2 노트북에서 `%%writefile pmco_core_v2.py`를 실행해도 모듈을 만들 수 있습니다.

In [ ]:
!pip -q install numpy pillow pandas requests beautifulsoup4

from pathlib import Path
from IPython.display import display
from google.colab import files

needed = ["pmco_core_v2.py", "pmco_evaluation.csv"]
missing = [name for name in needed if not Path(name).exists()]
if missing:
    print("다음 파일을 업로드해 주세요:", missing)
    files.upload()
if any(not Path(name).exists() for name in needed):
    raise FileNotFoundError("두 파일이 모두 있어야 합니다: " + ", ".join(needed))
print("준비 완료:", needed)

## ①-2. 가중치 프로필 실행·기존 CSV 재현 검증

- `baseline`: 복잡도 0.40, 채도 0.35, 대비 0.25
- `complexity_experiment`: 0.50, 0.20, 0.30
- `contrast_experiment`: 0.35, 0.25, 0.40

동일한 사진에서 에너지·BPM·장르가 어떻게 달라지는지 비교합니다. **정확도 평가는 아닙니다.**

In [ ]:
import numpy as np
import pandas as pd
from pmco_core_v2 import ImageFeatures, WEIGHT_PROFILES, map_music_profile

original = pd.read_csv("pmco_evaluation.csv", encoding="utf-8-sig")
required = ["filename", "brightness", "saturation", "contrast", "complexity", "warmth", "energy"]
absent = [c for c in required if c not in original.columns]
if absent:
    raise ValueError(f"CSV에 필요한 열이 없습니다: {absent}")

baseline = original.copy()
if "weight_profile" in baseline:
    baseline = baseline[baseline["weight_profile"].fillna("baseline") == "baseline"].copy()
baseline = baseline.drop_duplicates("filename", keep="last")

rows = []
for r in baseline.itertuples(index=False):
    f = ImageFeatures(
        brightness=float(r.brightness), saturation=float(r.saturation),
        contrast=float(r.contrast), complexity=float(r.complexity), warmth=float(r.warmth)
    )
    for profile in WEIGHT_PROFILES:
        m = map_music_profile(f, profile)
        rows.append({"filename": r.filename, "profile": profile,
                     "complexity": f.complexity, "saturation": f.saturation, "contrast": f.contrast,
                     "energy": m.energy, "bpm": m.bpm, "genre": m.genre})

comparison = pd.DataFrame(rows)
backcheck = comparison[comparison["profile"] == "baseline"].merge(
    baseline[["filename", "energy"]], on="filename", suffixes=("_recomputed", "_csv"))
max_diff = (backcheck["energy_recomputed"] - backcheck["energy_csv"]).abs().max()
print("저장된 baseline 데이터 수:", len(baseline))
print("baseline 재현 최대 오차:", round(float(max_diff), 10))
if max_diff > 1e-5:
    print("주의: 원래 CSV 값과 새 계산이 다릅니다. 가중치/원본 데이터 확인 필요")

print("\n프로필별 특징")
display(comparison.pivot(index="filename", columns="profile", values="energy").round(4))
display(comparison[["filename", "profile", "energy", "bpm", "genre"]])
comparison.to_csv("pmco_v2_profiles_checked.csv", index=False, encoding="utf-8-sig")
print("저장: pmco_v2_profiles_checked.csv (원본 CSV 변경 없음)")

### ①-3. 결과 해석

1. `baseline 재현 최대 오차`가 거의 0인지 확인.
2. 가중치를 바꿨을 때 특정 사진에서 `genre`가 변경되는지 확인.
3. *복잡도 가중치 증가*는 복잡도 자체가 낮은 사진에서 오히려 에너지를 낮출 수도 있습니다. 특정 가중치를 키운다고 모든 사진의 에너지가 올라가는 건 아닙니다.
4. 사진 6장에는 사람 평가가 없으므로 **어떤 프로필이 더 좋은지는 아직 알 수 없습니다.**

## ②-1. Wikimedia Commons 이미지 메타데이터 수집

처음에는 **메타데이터 최대 60건**을 조회합니다. 결과는 `commons_metadata_v3.csv`에 저장됩니다.

`CATEGORY`는 실제 Wikimedia Commons의 카테고리 이름입니다. 이 코드는 지정한 카테고리의 **직접 포함된 파일만 조회**하며 하위 카테고리는 재귀 탐색하지 않습니다.

Wikimedia의 API 속도 제한을 지키고, `User-Agent`에 프로젝트 링크를 명시합니다. 429/서버 오류는 짧게 재시도합니다. 자료마다 라이선스가 다르므로 공개 재사용 전 원본 페이지를 확인하세요.

In [ ]:
import requests, time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup

CATEGORY = "Category:Landscapes"
MAX_METADATA = 60   # 우선 60개. 더 많이 필요하면 점진적으로 증가

USER_AGENT = (
    "PMCO-KNU-Research/0.3 "
    "(https://github.com/wkwkwwk0712-debug/kkyu-repo)"
)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})
retry = Retry(total=2, backoff_factor=1.5,
              status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=["GET"], respect_retry_after_header=True)
session.mount("https://", HTTPAdapter(max_retries=retry))

base_params = {
    "action": "query", "format": "json",
    "generator": "categorymembers", "gcmtitle": CATEGORY,
    "gcmtype": "file", "gcmlimit": 50,
    "prop": "imageinfo", "iiprop": "url|extmetadata", "iiurlwidth": 512,
    "maxlag": 5,
}

records = {}  # 중복 파일 제목 제거
continuation = {}
for page_number in range(100):  # 실수로 무한 반복하지 않도록 상한 설정
    response = session.get(
        "https://commons.wikimedia.org/w/api.php",
        params={**base_params, **continuation}, timeout=35
    )
    response.raise_for_status()
    payload = response.json()
    if "error" in payload:
        raise RuntimeError(f"Wikimedia API 오류: {payload['error']}")

    pages = payload.get("query", {}).get("pages", {})
    for page in pages.values():
        title = str(page.get("title", ""))
        if not title.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            continue
        info = (page.get("imageinfo") or [{}])[0]
        thumb = info.get("thumburl")
        if not thumb:
            continue
        meta = info.get("extmetadata") or {}
        artist_html = (meta.get("Artist") or {}).get("value", "")
        records[title] = {
            "filename": "Commons:" + title,
            "file_title": title,
            "category": CATEGORY,
            "source": "Wikimedia Commons",
            "source_url": thumb,
            "source_page": info.get("descriptionurl", ""),
            "artist": BeautifulSoup(artist_html, "html.parser").get_text(" ", strip=True),
            "license": (meta.get("LicenseShortName") or {}).get("value", ""),
        }
        if len(records) >= MAX_METADATA:
            break

    continuation = payload.get("continue") or {}
    if len(records) >= MAX_METADATA or not continuation:
        break
    time.sleep(1.0)

commons_metadata = pd.DataFrame(list(records.values()), columns=[
    "filename", "file_title", "category", "source", "source_url",
    "source_page", "artist", "license"])
commons_metadata.to_csv("commons_metadata_v3.csv", index=False, encoding="utf-8-sig")
print("수집한 메타데이터 수:", len(commons_metadata))
display(commons_metadata.head(10))

## ②-2. 메타데이터에 연결된 이미지를 분석

`MAX_ANALYZE`만큼 작은 썸네일을 다운로드하여 **기존 `extract_features`와 `map_music_profile('baseline')`** 으로 분석합니다.

- `pmco_commons_features_v3.csv`: 크롤링 사진의 특징·음악 매핑 결과와 출처 정보
- `pmco_evaluation_combined_v3.csv`: **기존 수동 사진 6장 + 신규 크롤링 이미지** 합본
- 원래 `pmco_evaluation.csv`는 수정하지 않습니다.

**주의:** 저작권/라이선스·이미지 접근 여부를 개별 확인하고, 접근 실패한 파일은 건너뜁니다. 처음은 20장부터 테스트하세요.

In [ ]:
from io import BytesIO
from PIL import Image, ImageOps, UnidentifiedImageError
from urllib.parse import urlparse
from pmco_core_v2 import extract_features, plan_record, build_music_prompt

MAX_ANALYZE = 20  # 성공 후 50, 100 등으로 단계적으로 확대
IMAGE_BYTES_LIMIT = 6_000_000  # 썸네일 크기 제한
Image.MAX_IMAGE_PIXELS = 25_000_000

if commons_metadata.empty:
    raise ValueError("메타데이터가 비어 있습니다. ②-1 결과를 확인하세요.")

# 별도 출력 파일을 쓰므로 원본 데이터는 보존
new_rows, failed = [], []
for _, meta in commons_metadata.head(MAX_ANALYZE).iterrows():
    url = str(meta["source_url"])
    try:
        if urlparse(url).hostname != "upload.wikimedia.org":
            raise ValueError("공식 Wikimedia 업로드 도메인이 아님")
        with session.get(url, timeout=30, stream=True) as resp:
            resp.raise_for_status()
            chunks, size = [], 0
            for chunk in resp.iter_content(chunk_size=65536):
                size += len(chunk)
                if size > IMAGE_BYTES_LIMIT:
                    raise ValueError("파일 크기 제한 초과")
                chunks.append(chunk)
        with Image.open(BytesIO(b"".join(chunks))) as im:
            im = ImageOps.exif_transpose(im)
            features = extract_features(im)
        m = map_music_profile(features, "baseline")
        row = plan_record(str(meta["filename"]), features, m)
        row.update({
            "prompt": build_music_prompt(m),
            "weight_profile": "baseline",
            "source": "Wikimedia Commons", "category": meta["category"],
            "source_page": meta["source_page"], "source_url": url,
            "artist": meta["artist"], "license": meta["license"],
            "human_energy_1to5": pd.NA,
            "human_calmness_1to5": pd.NA,
            "human_music_fit_1to5": pd.NA,
            "notes": "",
        })
        new_rows.append(row)
    except (requests.RequestException, OSError, ValueError, UnidentifiedImageError) as exc:
        failed.append({"filename": meta["filename"], "reason": str(exc)[:160]})
    time.sleep(0.8)

commons_features = pd.DataFrame(new_rows)
commons_features.to_csv("pmco_commons_features_v3.csv", index=False, encoding="utf-8-sig")

manual = original.copy()  # ①에서 읽은 기존 CSV
if "source" not in manual.columns:
    manual["source"] = "manual upload"
if "weight_profile" not in manual.columns:
    manual["weight_profile"] = "baseline"
combined = pd.concat([manual, commons_features], ignore_index=True)
combined = combined.drop_duplicates(subset=["filename", "weight_profile"], keep="first")
combined.to_csv("pmco_evaluation_combined_v3.csv", index=False, encoding="utf-8-sig")

print("기존 수동 이미지:", len(manual))
print("신규 분석 성공:", len(commons_features), "실패:", len(failed))
print("통합 CSV:", len(combined), "행")
if failed:
    display(pd.DataFrame(failed).head(10))
display(combined[["filename", "source", "complexity", "saturation", "contrast", "energy", "bpm"]].tail())

### ②-3. KNIME 분석 방법 — 두 갈래

**A. 메타데이터 분석 (자료 출처·라이선스 파악)**

`CSV Reader(commons_metadata_v3.csv) → Column Filter(license, artist, category) → Missing Value(선택) → GroupBy(license별 Count) → Bar Chart`

**B. 이미지·음악 매핑 분석 (핵심 분석)**

`CSV Reader(pmco_evaluation_combined_v3.csv) → Column Filter(complexity, saturation, contrast, energy, bpm, genre, source) → Scatter Plot / Statistics View / GroupBy`

두 파일을 `filename` 기준으로 Joiner 연결할 수도 있지만, **통합 분석 CSV에는 이미 출처 정보가 포함**돼 있으므로 지금은 Joiner가 필수는 아닙니다.

※ 컬럼 형식이 문자열로 인식되면 KNIME의 CSV Reader 설정에서 해당 열을 숫자형으로 바꾸세요. 사람이 평가하지 않은 빈칸은 숫자 0으로 채우지 마세요.

## ③-1. 사람 평가용 입력 양식 생성

`pmco_human_rating_template_v3.csv`를 내려받은 뒤, 평가자가 **실제 사진을 본 다음** `human_energy_1to5`에 1~5 정수를 적도록 합니다.

- 1 매우 고요함 · 2 다소 고요함 · 3 보통 · 4 다소 역동적 · 5 매우 역동적
- `human_music_fit_1to5`는 실제 **생성 음악을 들었을 때만** 작성합니다.
- 사람 평가자마다 `rater_id`를 다르게 적으면 여러 사람의 평가 평균을 구할 수 있습니다.
- 각 파일명/링크가 맞는지 확인하세요. **파일명만 보고 채점하지 마세요.**
- 가능하면 고요한 사진·보통·활기찬 사진을 고르게 포함한 30장 이상, 2명 이상 평가를 권합니다.

In [ ]:
from pathlib import Path

# 크롤링 단계를 아직 실행하지 않았다면 기존 6장으로 양식 생성
rating_base = combined if "combined" in globals() else original
rating_base = rating_base.drop_duplicates("filename", keep="first")

sheet_cols = ["filename", "source_page", "source_url"]
for col in sheet_cols:
    if col not in rating_base:
        rating_base[col] = ""

# 점수를 보지 않고 평가하도록 prediction/energy 컬럼은 제외합니다.
template = rating_base[sheet_cols].copy()
template.insert(1, "rater_id", "R1")
template["human_energy_1to5"] = pd.NA
template["human_calmness_1to5"] = pd.NA
template["human_music_fit_1to5"] = pd.NA
template["rating_notes"] = ""
template.to_csv("pmco_human_rating_template_v3.csv", index=False, encoding="utf-8-sig")
print("평가 양식 행 수:", len(template))
display(template.head())
print("다운로드: 왼쪽 파일 창 또는 아래 셀 활용")

## ③-2. 평가점수를 입력했다면 가중치 학습·검증

**반드시 사람 점수 입력 후 실행하세요.** 점수가 비어 있으면 안내만 하고 중단합니다.

평가자는 이미지 자체의 역동성을 평가하므로 이 단계는 **이미지 특징 → 음악 에너지 매핑**을 검증합니다. 실제 음악 적합도는 별도의 평가입니다.

- 점수 1~5를 `0, .25, .50, .75, 1.0`에 대응시킵니다.
- 사진 단위로 평가자 평균을 낸 후, 무작위로 훈련 75% / 검증 25% 분리합니다.
- 훈련 사진으로만 0.05 간격 가중치 후보를 탐색합니다. 검증 사진은 학습에 쓰지 않습니다.
- **검증 MAE**가 기준 가중치보다 실제로 낮을 때만 개선 가능성을 이야기합니다.
- 표본이 작으면 수치는 참고용일 뿐, 최적 가중치라고 확정하지 않습니다.

In [ ]:
from itertools import product
import numpy as np
import pandas as pd
from pathlib import Path

# 사용자가 점수를 채워 업로드한 파일명으로 수정 가능
RATINGS = Path("pmco_human_rating_template_v3.csv")
if not RATINGS.exists():
    raise FileNotFoundError("평가 양식 CSV를 찾지 못했습니다. 먼저 업로드하세요.")

ratings = pd.read_csv(RATINGS, encoding="utf-8-sig")
ratings["human_energy_1to5"] = pd.to_numeric(ratings["human_energy_1to5"], errors="coerce")
ratings = ratings.dropna(subset=["human_energy_1to5"])
if not ratings["human_energy_1to5"].between(1, 5).all():
    raise ValueError("human_energy_1to5에는 1~5의 점수만 입력해 주세요.")

labels = ratings.groupby("filename", as_index=False)["human_energy_1to5"].mean()
feature_base = combined if "combined" in globals() else original
feature_base = feature_base.copy()
if "weight_profile" in feature_base:
    feature_base = feature_base[feature_base["weight_profile"].fillna("baseline") == "baseline"]
feature_base = feature_base.drop_duplicates("filename", keep="first")

labeled = feature_base.merge(labels, on="filename", how="inner", suffixes=("", "_human"))
# 기존 CSV에 비어 있는 human_energy_1to5 열이 이미 있어 병합하면 접미사가 생깁니다.
if "human_energy_1to5_human" in labeled:
    labeled["human_energy_1to5"] = labeled["human_energy_1to5_human"]

N = len(labeled)
print("점수 있는 고유 사진 수:", N)
if N < 12:
    print("가중치 학습은 보류: 서로 다른 사진 12장 이상 평가한 뒤 다시 실행하세요. 가능하면 30장 이상 권장.")
else:
    X = labeled[["complexity", "saturation", "contrast"]].to_numpy(dtype=float)
    y = (labeled["human_energy_1to5"].to_numpy(dtype=float) - 1.0) / 4.0
    if not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError("특징/평가값에 결측치나 무한대가 있습니다.")
    rng = np.random.default_rng(2026)
    order = rng.permutation(N)
    cut = max(2, int(round(N * 0.75)))
    cut = min(cut, N - 3)  # 검증용 사진 최소 3장
    train_idx, test_idx = order[:cut], order[cut:]
    # 0.05 간격, 음수 불가, 합 1.0인 가중치만 비교
    candidates = []
    for a in range(21):
        for b in range(21 - a):
            c = 20 - a - b
            candidates.append((a / 20, b / 20, c / 20))
    W = np.asarray(candidates)
    train_error = np.mean(np.abs(X[train_idx] @ W.T - y[train_idx, None]), axis=0)
    # 동점에 가까우면 기존 가중치에 가까운 쪽 선택하여 불필요한 변경 억제
    baseline_w = np.array([0.40, 0.35, 0.25])
    regularized = train_error + 0.005 * np.linalg.norm(W - baseline_w, axis=1)
    best = W[int(np.argmin(regularized))]
    experiments = {"baseline": baseline_w,
                   "complexity_experiment": np.array([0.50, 0.20, 0.30]),
                   "contrast_experiment": np.array([0.35, 0.25, 0.40]),
                   "learned_candidate": best}
    summary = []
    for name, weights in experiments.items():
        pred = X @ weights
        summary.append({
            "profile": name,
            "w_complexity": weights[0], "w_saturation": weights[1], "w_contrast": weights[2],
            "train_MAE": np.mean(np.abs(pred[train_idx] - y[train_idx])),
            "holdout_MAE": np.mean(np.abs(pred[test_idx] - y[test_idx]))
        })
    out = pd.DataFrame(summary)
    out.to_csv("pmco_weight_validation_v3.csv", index=False, encoding="utf-8-sig")
    print(f"훈련 {len(train_idx)}장 / 검증 {len(test_idx)}장")
    display(out.round(4))
    print("학습으로 찾은 후보 가중치:", tuple(np.round(best, 3)))
    print("주의: 검증 MAE를 비교하고, 새로운 사진으로 반복 검증 후 채택 여부를 결정하세요.")

## 파일 다운로드 및 GitHub 커밋 기록

`파일 다운로드`는 Colab의 왼쪽 📁에서 원하는 CSV를 우클릭하거나 다음 코드로 실행할 수 있습니다.

- 1차: `pmco_v2_profiles_checked.csv` — **가중치 코드 확인**
- 2차: `commons_metadata_v3.csv`, `pmco_commons_features_v3.csv`, `pmco_evaluation_combined_v3.csv` — **크롤링·특징 데이터 확장**
- 3차: `pmco_human_rating_template_v3.csv`, `pmco_weight_validation_v3.csv` — **평가 및 검증**

추천 커밋 분리:
1. `Verify PMCO v2 weight profiles against baseline data`
2. `Add Commons image metadata and feature extraction pipeline`
3. `Add human evaluation protocol and weight validation experiment`

워크플로우/코드 변경을 먼저 커밋하고, 실험 CSV는 개인정보·저작권 출처 정보를 확인한 후 업로드하세요.

In [ ]:
from google.colab import files
from pathlib import Path

# 원하는 파일명 하나를 넣어 수동 다운로드
DOWNLOAD_FILE = "pmco_v2_profiles_checked.csv"
if Path(DOWNLOAD_FILE).exists():
    files.download(DOWNLOAD_FILE)
else:
    print("아직 생성되지 않은 파일입니다:", DOWNLOAD_FILE)